In [ ]:
pip install -U langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Define the document variable with the text you want to split
document = """This is a sample document that needs to be split into smaller chunks. LangChain's RecursiveCharacterTextSplitter is useful for this purpose. It tries to split text by different characters to keep sentences and paragraphs together for as long as possible."""

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
texts = text_splitter.split_text(document)

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=100, chunk_overlap=0
)
texts = text_splitter.split_text(document)

In [ ]:
import re
import json
from pathlib import Path
from google.colab import files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# =====================================================================
# 1. DEFINIÇÃO DAS 10 FUNÇÕES DE CHUNKING
# =====================================================================

def chunk_fixo_200(texto, nome_arquivo):
    chunk_size = 200
    trechos, metadados = [], []
    for i in range(0, len(texto), chunk_size):
        chunk = texto[i:i + chunk_size]
        if chunk.strip():
            trechos.append(chunk)
            metadados.append({"arquivo": nome_arquivo, "estrategia": "fixo_200", "chunk_id": f"char_{i}"})
    return trechos, metadados

def chunk_fixo_500(texto, nome_arquivo):
    chunk_size = 500
    trechos, metadados = [], []
    for i in range(0, len(texto), chunk_size):
        chunk = texto[i:i + chunk_size]
        if chunk.strip():
            trechos.append(chunk)
            metadados.append({"arquivo": nome_arquivo, "estrategia": "fixo_500", "chunk_id": f"char_{i}"})
    return trechos, metadados

def chunk_fixo_1000(texto, nome_arquivo):
    chunk_size = 1000
    trechos, metadados = [], []
    for i in range(0, len(texto), chunk_size):
        chunk = texto[i:i + chunk_size]
        if chunk.strip():
            trechos.append(chunk)
            metadados.append({"arquivo": nome_arquivo, "estrategia": "fixo_1000", "chunk_id": f"char_{i}"})
    return trechos, metadados

def chunk_fixo_2000(texto, nome_arquivo):
    chunk_size = 2000
    trechos, metadados = [], []
    for i in range(0, len(texto), chunk_size):
        chunk = texto[i:i + chunk_size]
        if chunk.strip():
            trechos.append(chunk)
            metadados.append({"arquivo": nome_arquivo, "estrategia": "fixo_2000", "chunk_id": f"char_{i}"})
    return trechos, metadados

def chunk_fixo_500_overlap_50(texto, nome_arquivo):
    chunk_size, overlap = 500, 50
    step = chunk_size - overlap
    trechos, metadados = [], []
    i, chunk_num = 0, 0
    while i < len(texto):
        chunk = texto[i:i + chunk_size]
        if chunk.strip():
            trechos.append(chunk)
            metadados.append({"arquivo": nome_arquivo, "estrategia": "fixo_500_ovlp_50", "chunk_id": chunk_num})
        i += step
        chunk_num += 1
    return trechos, metadados

def chunk_fixo_500_overlap_200(texto, nome_arquivo):
    chunk_size, overlap = 500, 200
    step = chunk_size - overlap
    trechos, metadados = [], []
    i, chunk_num = 0, 0
    while i < len(texto):
        chunk = texto[i:i + chunk_size]
        if chunk.strip():
            trechos.append(chunk)
            metadados.append({"arquivo": nome_arquivo, "estrategia": "fixo_500_ovlp_200", "chunk_id": chunk_num})
        i += step
        chunk_num += 1
    return trechos, metadados

def chunk_por_paragrafo(texto, nome_arquivo):
    paragrafos = re.split(r'\n\s*\n', texto)
    trechos, metadados = [], []
    for i, paragrafo in enumerate(paragrafos, 1):
        paragrafo = paragrafo.strip()
        if paragrafo:
            trechos.append(paragrafo)
            metadados.append({"arquivo": nome_arquivo, "estrategia": "paragrafo", "paragrafo_id": i, "tamanho": len(paragrafo)})
    return trechos, metadados

def chunk_por_sentencas_3(texto, nome_arquivo):
    sentencas = re.split(r'(?<=[.!?])\s+', texto)
    sentencas = [s.strip() for s in sentencas if s.strip()]
    trechos, metadados = [], []
    chunk_num = 0
    for i in range(0, len(sentencas), 3):
        grupo = sentencas[i:i + 3]
        chunk = " ".join(grupo)
        if chunk.strip():
            trechos.append(chunk)
            metadados.append({"arquivo": nome_arquivo, "estrategia": "sentencas_3", "chunk_id": chunk_num, "tamanho": len(chunk)})
            chunk_num += 1
    return trechos, metadados

def chunk_recursivo(texto, nome_arquivo, chunk_size=500, chunk_overlap=0):
    separadores = ["\n\n", "\n", ". ", " ", ""]

    def _split_recursivo(texto_atual, separadores_restantes):
        if len(texto_atual) <= chunk_size:
            return [texto_atual] if texto_atual.strip() else []

        separador = separadores_restantes[0] if separadores_restantes else ""
        prox_separadores = separadores_restantes[1:] if len(separadores_restantes) > 1 else []

        if separador == "":
            chunks = []
            for i in range(0, len(texto_atual), chunk_size - chunk_overlap if chunk_overlap > 0 else chunk_size):
                chunk = texto_atual[i:i + chunk_size]
                if chunk.strip():
                    chunks.append(chunk)
            return chunks

        partes = texto_atual.split(separador)
        chunks_final, chunk_atual = [], ""

        for parte in partes:
            tentativa = chunk_atual + (separador if chunk_atual else "") + parte
            if len(tentativa) <= chunk_size:
                chunk_atual = tentativa
            else:
                if chunk_atual.strip():
                    if len(parte) > chunk_size and prox_separadores:
                        chunks_final.append(chunk_atual)
                        chunks_final.extend(_split_recursivo(parte, prox_separadores))
                        chunk_atual = ""
                    else:
                        chunks_final.append(chunk_atual)
                        chunk_atual = parte
                else:
                    chunk_atual = parte

        if chunk_atual.strip():
            chunks_final.append(chunk_atual)
        return chunks_final

    chunks = _split_recursivo(texto, separadores)
    trechos, metadados = [], []
    for i, chunk in enumerate(chunks):
        if chunk.strip():
            trechos.append(chunk)
            metadados.append({"arquivo": nome_arquivo, "estrategia": "recursivo", "chunk_id": i, "tamanho": len(chunk)})
    return trechos, metadados

def chunk_por_heading_markdown(texto, nome_arquivo):
    linhas = texto.split("\n")
    trechos, metadados = [], []
    chunk_atual, heading_atual, nivel_atual = "", "(início do documento)", 0

    for linha in linhas:
        match_heading = re.match(r'^(#{1,6})\s+(.+)', linha)
        if match_heading:
            if chunk_atual.strip():
                trechos.append(chunk_atual.strip())
                metadados.append({"arquivo": nome_arquivo, "estrategia": "heading_md", "heading": heading_atual, "nivel": nivel_atual, "tamanho": len(chunk_atual.strip())})
            nivel_atual = len(match_heading.group(1))
            heading_atual = match_heading.group(2).strip()
            chunk_atual = linha + "\n"
        else:
            chunk_atual += linha + "\n"

    if chunk_atual.strip():
        trechos.append(chunk_atual.strip())
        metadados.append({"arquivo": nome_arquivo, "estrategia": "heading_md", "heading": heading_atual, "nivel": nivel_atual, "tamanho": len(chunk_atual.strip())})
    return trechos, metadados

# =====================================================================
# 2. CARREGAMENTO DOS ARQUIVOS E BUSCA SEMÂNTICA
# =====================================================================

documentos = {}
for arq in Path('.').glob('*.md'):
    with open(arq, "r", encoding="utf-8") as f:
        documentos[arq.name] = f.read()

def busca_semantica(consulta, trechos, metadados, top_k=2):
    if not trechos:
        return []
    vectorizer = TfidfVectorizer()
    matriz_tfidf = vectorizer.fit_transform(trechos + [consulta])
    vetor_consulta = matriz_tfidf[-1]
    vetores_documentos = matriz_tfidf[:-1]
    similaridades = cosine_similarity(vetor_consulta, vetores_documentos)[0]
    indices_top = similaridades.argsort()[::-1][:top_k]

    resultados = []
    for idx in indices_top:
        resultados.append({
            "texto": trechos[idx],
            "metadados": metadados[idx],
            "similaridade": float(similaridades[idx])
        })
    return resultados

# =====================================================================
# 3. MAPEAMENTO E EXECUÇÃO DO BENCHMARK
# =====================================================================

estrategias = {
    "1. Fixo 200":       lambda t, n: chunk_fixo_200(t, n),
    "2. Fixo 500":       lambda t, n: chunk_fixo_500(t, n),
    "3. Fixo 1000":      lambda t, n: chunk_fixo_1000(t, n),
    "4. Fixo 2000":      lambda t, n: chunk_fixo_2000(t, n),
    "5. 500 + ovlp 50":  lambda t, n: chunk_fixo_500_overlap_50(t, n),
    "6. 500 + ovlp 200": lambda t, n: chunk_fixo_500_overlap_200(t, n),
    "7. Parágrafos":     lambda t, n: chunk_por_paragrafo(t, n),
    "8. 3 Sentenças":    lambda t, n: chunk_por_sentencas_3(t, n),
    "9. Recursivo":      lambda t, n: chunk_recursivo(t, n, chunk_size=500, chunk_overlap=0),
    "10. Headings MD":   lambda t, n: chunk_por_heading_markdown(t, n),
}

consultas_teste = [
    "inteligência artificial e ética",
    "algoritmo e redes sociais",
    "escrita acadêmica"
]

resultado_consolidado = []

for consulta in consultas_teste:
    print(f"\n🔍 CONSULTA: \"{consulta}\"")
    print(f"{'Estratégia':<22} {'#1 sim':>8} {'#1 tam':>8} {'Chunks':>8}")

    dados_consulta = {"consulta": consulta, "resultados": []}

    for nome_est, func in estrategias.items():
        todos_trechos, todos_meta = [], []
        for nome_doc, conteudo in documentos.items():
            t, m = func(conteudo, nome_doc)
            todos_trechos.extend(t)
            todos_meta.extend(m)

        resultados = busca_semantica(consulta, todos_trechos, todos_meta, top_k=2)
        tam1 = len(resultados[0]["texto"]) if resultados else 0
        sim1 = resultados[0]["similaridade"] if resultados else 0
        print(f"{nome_est:<22} {sim1:>8.4f} {tam1:>8} {len(todos_trechos):>8}")

        dados_consulta["resultados"].append({
            "estrategia": nome_est,
            "total_chunks": len(todos_trechos),
            "top1_similaridade": round(sim1, 4),
            "top1_tamanho": tam1,
            "top1_texto": resultados[0]["texto"] if resultados else ""
        })

    resultado_consolidado.append(dados_consulta)

# Salva e faz o download automático do JSON
with open("resultado_consolidado.json", "w", encoding="utf-8") as f:
    json.dump(resultado_consolidado, f, ensure_ascii=False, indent=4)

print("\n✅ Processo finalizado com sucesso!")
files.download("resultado_consolidado.json")


🔍 CONSULTA: "inteligência artificial e ética"
Estratégia               #1 sim   #1 tam   Chunks
1. Fixo 200              0.3078      200      742
2. Fixo 500              0.3562      500      298
3. Fixo 1000             0.2870     1000      150
4. Fixo 2000             0.2573     2000       76
5. 500 + ovlp 50         0.3545      500      330
6. 500 + ovlp 200        0.3544      500      496
7. Parágrafos            0.5773       76      447
8. 3 Sentenças           0.3291      597      378
9. Recursivo             0.3374      349      287
10. Headings MD          0.3891      116       67

🔍 CONSULTA: "algoritmo e redes sociais"
Estratégia               #1 sim   #1 tam   Chunks
1. Fixo 200              0.2273      200      742
2. Fixo 500              0.1791      500      298
3. Fixo 1000             0.1644     1000      150
4. Fixo 2000             0.1136     2000       76
5. 500 + ovlp 50         0.2181      500      330
6. 500 + ovlp 200        0.2135      500      496
7. Parágrafo

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>